# 06b - Compression Diagnostic

The compression sweep results are suspiciously high (all compress_* conditions
**outperform** self_prefill at 0.803). This notebook diagnoses why.

Hypotheses to check:
1. **Answer leakage**: The paraphraser includes the final numeric answer in its output
2. **Think-tag contamination**: Qwen3-8B wraps output in `<think>` tags, injecting extra reasoning
3. **"The answer is" duplication**: Compressed text already contains "The answer is: X",
   so the prefill prompt becomes `...The answer is: 42\nThe answer is: ` (trivial extraction)
4. **Lack of actual compression**: Paraphraser just rewrites at equal or greater length
5. **Paraphraser solves the problem itself**: Instead of compressing, the 8B model re-derives
   the answer (possibly more accurately than the original 4B COT)

In [ ]:
import subprocess, sys
from pathlib import Path

WORKSPACE = Path("/workspace/13-4-2026")
REPO_DIR = WORKSPACE / "legibility"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "-b", "13-4-2026",
        "https://github.com/JackHopkins/legibility.git",
        str(REPO_DIR)
    ], check=True)

sys.path.insert(0, str(REPO_DIR))
from lib.config import *

for d in [CACHE_DIR, COT_CACHE, PARAPHRASE_CACHE, PREFILL_CACHE, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
import json
import re
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# Load original COTs
cots = {}
for p in COT_CACHE.glob("*.json"):
    r = json.loads(p.read_text())
    cots[r["problem_id"]] = r
print(f"Loaded {len(cots)} original COTs")

# Load all compression conditions
LEVELS = ["light", "medium", "heavy", "ultra_heavy", "single_sentence"]

compressions = {}
for level in LEVELS:
    # 06 used compress_{level} for new levels, paraphrase_light/heavy for existing
    if level == "light":
        prefix = "paraphrase_light"
    elif level == "heavy":
        prefix = "paraphrase_heavy"
    else:
        prefix = f"compress_{level}"

    data = {}
    for p in PARAPHRASE_CACHE.glob(f"{prefix}_*.json"):
        r = json.loads(p.read_text())
        data[r["problem_id"]] = r
    compressions[level] = data
    print(f"{level} ({prefix}): {len(data)} cached")

## Check 1: Think-tag contamination

Does the paraphraser output contain `<think>` tags? If so, the compressed text
includes the model's own reasoning, not just a compression of the input.

In [ ]:
for level in LEVELS:
    data = compressions[level]
    n_think = 0
    n_total = 0
    for pid, r in data.items():
        text = r["paraphrased_cot"]
        n_total += 1
        if "<think>" in text or "</think>" in text:
            n_think += 1
    pct = (n_think / n_total * 100) if n_total else 0
    flag = " *** CONTAMINATED ***" if n_think > 0 else ""
    print(f"{level:20s}: {n_think}/{n_total} contain <think> tags ({pct:.1f}%){flag}")

## Check 2: Answer leakage

Does the compressed text contain the gold answer as a standalone number?
Also check if it contains "The answer is" or similar answer-revealing patterns.

In [ ]:
for level in LEVELS:
    data = compressions[level]
    n_has_answer_phrase = 0
    n_has_gold_number = 0
    n_has_boxed = 0
    n_total = 0

    for pid, r in data.items():
        text = r["paraphrased_cot"]
        gold = cots[pid]["gold_answer"]
        n_total += 1

        # Check for "The answer is" pattern
        if re.search(r"[Tt]he answer is:?\s*-?\d", text):
            n_has_answer_phrase += 1

        # Check for gold answer as final number
        numbers = re.findall(r"-?\d[\d,]*", text)
        if numbers and gold is not None:
            last_num = int(numbers[-1].replace(",", ""))
            if last_num == gold:
                n_has_gold_number += 1

        # Check for \boxed{} or #### patterns
        if "####" in text or "\\boxed" in text:
            n_has_boxed += 1

    print(f"\n--- {level} ---")
    print(f"  'The answer is: N' pattern: {n_has_answer_phrase}/{n_total} ({n_has_answer_phrase/n_total*100:.1f}%)")
    print(f"  Last number = gold answer:  {n_has_gold_number}/{n_total} ({n_has_gold_number/n_total*100:.1f}%)")
    print(f"  Contains #### or \\boxed:   {n_has_boxed}/{n_total} ({n_has_boxed/n_total*100:.1f}%)")

## Check 3: Actual compression ratios

Are the compressions actually shorter? If the "compressed" version is the same
length or longer, the paraphraser is re-deriving rather than compressing.

In [ ]:
fig, axes = plt.subplots(1, len(LEVELS), figsize=(4 * len(LEVELS), 4), sharey=True)

for ax, level in zip(axes, LEVELS):
    data = compressions[level]
    ratios = []
    orig_lens = []
    comp_lens = []
    for pid, r in data.items():
        if pid not in cots:
            continue
        orig_len = len(cots[pid]["cot_text"])
        comp_len = len(r["paraphrased_cot"])
        if orig_len > 0:
            ratios.append(comp_len / orig_len)
            orig_lens.append(orig_len)
            comp_lens.append(comp_len)

    ax.hist(ratios, bins=50, alpha=0.7, edgecolor="black", linewidth=0.3)
    ax.axvline(x=1.0, color="red", linestyle="--", alpha=0.5)
    ax.axvline(x=np.median(ratios), color="blue", linestyle="-", alpha=0.7,
               label=f"median={np.median(ratios):.2f}")
    ax.set_xlabel("Compression ratio (compressed / original)")
    ax.set_title(f"{level}")
    ax.legend(fontsize=8)

    n_longer = sum(1 for r in ratios if r > 1.0)
    print(f"{level:20s}: median ratio={np.median(ratios):.2f}, "
          f"mean={np.mean(ratios):.2f}, "
          f"longer than original: {n_longer}/{len(ratios)} ({n_longer/len(ratios)*100:.1f}%)")
    print(f"{'':20s}  orig chars: median={np.median(orig_lens):.0f}, "
          f"comp chars: median={np.median(comp_lens):.0f}")

axes[0].set_ylabel("Count")
plt.suptitle("Compression Ratios by Level", fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "compression_diagnostic_ratios.png", dpi=200, bbox_inches="tight")
plt.show()

## Check 4: What the prefill prompt actually looks like

Show the exact text being fed to the model for a few examples.
If the compressed COT contains the answer, the prefill becomes trivial.

In [ ]:
from lib.prompts import build_prefill_prompt

# Pick 3 problems to inspect
sample_pids = sorted(cots.keys())[:3]

for pid in sample_pids:
    orig = cots[pid]
    print(f"{'='*80}")
    print(f"Problem {pid}: gold_answer = {orig['gold_answer']}")
    print(f"Question: {orig['question'][:150]}...")
    print(f"{'='*80}")

    # Original COT
    print(f"\n--- Original COT ({len(orig['cot_text'])} chars) ---")
    print(orig["cot_text"][:400])
    if len(orig["cot_text"]) > 400:
        print(f"... [{len(orig['cot_text']) - 400} more chars]")

    for level in LEVELS:
        if pid in compressions[level]:
            comp = compressions[level][pid]["paraphrased_cot"]
            ratio = len(comp) / len(orig["cot_text"]) if len(orig["cot_text"]) > 0 else 0

            # Check for answer leakage
            has_answer = False
            if orig["gold_answer"] is not None:
                if re.search(rf"\b{orig['gold_answer']}\b", comp[-100:]):
                    has_answer = True

            leak_flag = " [ANSWER IN TEXT!]" if has_answer else ""
            print(f"\n--- {level} ({len(comp)} chars, ratio={ratio:.2f}){leak_flag} ---")
            print(comp[:400])
            if len(comp) > 400:
                print(f"... [{len(comp) - 400} more chars]")

            # Show the actual prefill that gets sent to the model
            prefill = build_prefill_prompt(orig["question"], comp, PRIMARY_MODEL)
            # Show just the tail (COT ending + "The answer is: ")
            print(f"\n  PREFILL TAIL: ...{prefill[-200:]}")
    print()

## Check 5: Does the compressed text contain the answer but the original COT doesn't?

If the original COT had the wrong answer, but the compression "fixes" it,
the paraphraser is solving the problem, not compressing.

In [ ]:
for level in LEVELS:
    data = compressions[level]
    n_orig_wrong_comp_right = 0
    n_orig_right_comp_wrong = 0
    n_both_right = 0
    n_both_wrong = 0
    n_total = 0

    for pid, r in data.items():
        if pid not in cots:
            continue
        gold = cots[pid]["gold_answer"]
        if gold is None:
            continue

        orig_text = cots[pid]["cot_text"]
        comp_text = r["paraphrased_cot"]
        n_total += 1

        # Check if gold answer appears as the last number in each
        def last_number_is_gold(text, gold):
            numbers = re.findall(r"-?\d[\d,]*", text)
            if numbers:
                return int(numbers[-1].replace(",", "")) == gold
            return False

        orig_has = last_number_is_gold(orig_text, gold)
        comp_has = last_number_is_gold(comp_text, gold)

        if orig_has and comp_has:
            n_both_right += 1
        elif not orig_has and comp_has:
            n_orig_wrong_comp_right += 1
        elif orig_has and not comp_has:
            n_orig_right_comp_wrong += 1
        else:
            n_both_wrong += 1

    print(f"\n--- {level} (last number = gold check) ---")
    print(f"  Both have gold as last num:    {n_both_right:4d} ({n_both_right/n_total*100:.1f}%)")
    print(f"  Orig wrong, comp 'fixed':      {n_orig_wrong_comp_right:4d} ({n_orig_wrong_comp_right/n_total*100:.1f}%) <-- SUSPICIOUS")
    print(f"  Orig right, comp lost:         {n_orig_right_comp_wrong:4d} ({n_orig_right_comp_wrong/n_total*100:.1f}%)")
    print(f"  Both missing gold:             {n_both_wrong:4d} ({n_both_wrong/n_total*100:.1f}%)")

## Check 6: Compare with the prefill results

For the cases where compress_X_self gets it right but self_prefill gets it wrong:
is the compressed text trivially giving away the answer?

In [ ]:
# Load prefill results for self_prefill and each compress level
def load_results(condition):
    results = {}
    for p in PREFILL_CACHE.glob(f"{condition}_*.json"):
        r = json.loads(p.read_text())
        results[r["problem_id"]] = r
    return results

self_results = load_results("self_prefill")

level_to_condition = {
    "light": "compress_light_self",
    "medium": "compress_medium_self",
    "heavy": "compress_heavy_self",
    "ultra_heavy": "compress_ultra_heavy_self",
    "single_sentence": "compress_single_sentence_self",
}

for level, cond in level_to_condition.items():
    comp_results = load_results(cond)
    if not comp_results:
        continue

    # Cases where self_prefill wrong, compress right
    flipped_to_right = []
    flipped_to_wrong = []
    for pid in comp_results:
        if pid not in self_results:
            continue
        self_correct = (self_results[pid]["predicted_answer"] == self_results[pid]["gold_answer"])
        comp_correct = (comp_results[pid]["predicted_answer"] == comp_results[pid]["gold_answer"])
        if not self_correct and comp_correct:
            flipped_to_right.append(pid)
        elif self_correct and not comp_correct:
            flipped_to_wrong.append(pid)

    print(f"\n--- {level} ({cond}) ---")
    print(f"  self_prefill wrong -> compress right: {len(flipped_to_right)} <-- SUSPICIOUS")
    print(f"  self_prefill right -> compress wrong: {len(flipped_to_wrong)}")
    print(f"  Net improvement: {len(flipped_to_right) - len(flipped_to_wrong)}")

    # Show a few flipped-to-right examples
    for pid in flipped_to_right[:3]:
        gold = cots[pid]["gold_answer"]
        comp_text = compressions.get(level, {}).get(pid, {}).get("paraphrased_cot", "N/A")
        self_pred = self_results[pid]["predicted_answer"]
        comp_pred = comp_results[pid]["predicted_answer"]
        print(f"\n    Problem {pid}: gold={gold}, self_pred={self_pred}, comp_pred={comp_pred}")
        print(f"    Compressed text (last 200 chars): ...{comp_text[-200:]}")

## Check 7: Token-level analysis of generated answers

What exactly does the model generate after "The answer is: " for each condition?
If the compressed text already trails off with the number, the model just copies it.

In [ ]:
# Distribution of generated token lengths per condition
all_conditions = ["self_prefill"] + list(level_to_condition.values())
all_labels = ["self_prefill"] + LEVELS

fig, ax = plt.subplots(figsize=(10, 4))

for i, (cond, label) in enumerate(zip(all_conditions, all_labels)):
    results = load_results(cond)
    if not results:
        continue
    gen_lens = [len(r["generated_tokens"]) for r in results.values()]
    ax.boxplot(gen_lens, positions=[i], widths=0.6)

ax.set_xticks(range(len(all_labels)))
ax.set_xticklabels(all_labels, rotation=15)
ax.set_ylabel("Generated token string length")
ax.set_title("Generated Answer Length by Condition", fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "compression_diagnostic_genlens.png", dpi=200, bbox_inches="tight")
plt.show()

# Show most common generated texts per condition
for cond, label in zip(all_conditions, all_labels):
    results = load_results(cond)
    if not results:
        continue
    gen_texts = [r["generated_tokens"][:50] for r in results.values()]
    most_common = Counter(gen_texts).most_common(5)
    print(f"\n{label} - most common generated texts:")
    for text, count in most_common:
        print(f"  {count:4d}x  '{text}'")

## Summary

In [ ]:
print("=" * 70)
print("COMPRESSION DIAGNOSTIC SUMMARY")
print("=" * 70)
print()
print("If you see:")
print("  - High % of <think> tags -> paraphraser is reasoning, not compressing")
print("  - High % of 'The answer is' in compressed text -> answer leakage")
print("  - Compression ratio > 1.0 -> paraphraser is expanding, not compressing")
print("  - Many 'orig wrong, comp fixed' -> paraphraser is re-solving the problem")
print("  - Many 'self wrong -> compress right' flips -> the compressed text")
print("    contains information the original COT didn't have")
print()
print("Likely fix: strip <think> tags from paraphraser output, add explicit")
print("instruction to NOT include the final answer, or post-process to remove")
print("any 'The answer is' suffix from compressed text before prefilling.")